# feder-kg master pipeline: publications → ontology-constrained knowledge graph

This notebook runs the whole document-processing pipeline with **Claude** as the extraction model.
It is a thin orchestrator: the logic lives in `src/` (`src/extract.py`, `src/ontology_loader.py`, `src/load_graph.py`
and helpers in `src/pipeline/`), so `python -m src.extract` runs the same thing without the notebook.

```
PDF ─► 1 ingest (Docling) ─► 2 structure ─► V visuals (Claude vision on table/figure crops) ─► 3 pass 1: samples & process history
                                                              ─► 4 pass 2: measurements & values
                                                              ─► 5 pass 3: other entities & relations
    ─► 6 validate (code checks) ─► 7 resolve ─► 8 export (graph.json / CSV / Turtle / Neo4j) ─► 9 evaluate
```

| Stage | Module | Idea taken from the course notebooks |
|---|---|---|
| Ingest + structure | `src/pipeline/docling_ingest.py` (Docling) | *Publication structure*: an explicit document model (title, sections, captions, tables, figures) with page provenance. Docling's layout model finds the regions and the reading order; section roles (methods/results/…) are added here |
| Cleaning | `src/pipeline/clean.py` | *Murari*: explicit, documented cleanup (ligatures, hyphenation), raw text kept |
| Ontology gap terms, search | `src/pipeline/triage.py` | *Murari*: RAKE-style phrases, BM25 |
| Extraction | `src/extract.py`, `src/pipeline/schemas.py`, `llm.py` | *Spangher*: output contract first, evidence quotes, polarity, positive + negative examples, rejected candidates, content-addressed cache |
| Tables / figures | `src/extract.py` (`run_visuals_docling`) | *Publication structure*: send only the hard regions to a vision model (table crops Docling couldn't structure; optional figure descriptions) |
| Validation | `src/pipeline/validate.py` | *Spangher*: quote anchoring without fuzzy repair; `structurally valid` ≠ `verified` |

What is new for this project: classes and relations are constrained by **your ontology** (`ontology/`, sliced by `ontology/slice_manifest.json`),
samples are tracked with their **process history**, values are **unit-normalized**, and there is an
**evaluation** step against your hand-annotated papers.

**Every stage saves `data/extractions/<paper_id>/<stage>.json` and is skipped on re-run if its inputs didn't change.
Every Claude response is cached by exact request, so re-running never pays twice.**

## 0. Setup (once)

1. From the repo root: `pip install -r requirements.txt`, then `docker compose up -d` if you want Neo4j.
2. Put your PDFs in `data/publications/` (not committed; their hashes go to `checksums.json`).
3. Put the ontology in `ontology/` (the file named in `slice_manifest.json` → `ontology_file`, by default
   `matsci-v10-semantic-closure.owl`). Until it is there, `ontology/starter_matsci_ontology.ttl` is used as a stand-in.
   Then check `ontology/slice_manifest.json` (which subtrees count as materials, processes, techniques, properties…).
4. In `.env` set `ANTHROPIC_API_KEY` (from console.anthropic.com). The key is never printed or saved.
5. Docling downloads its models from Hugging Face on first use (layout, TableFormer, figure classifier; a few hundred MB).
   To fetch them up front: `docling-tools models download layout tableformer picture_classifier`. Conversion runs on CPU
   (or a CUDA GPU if available); expect very roughly a few seconds per page on CPU.

In [ ]:
import sys, json, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
assert (PROJECT / "src" / "extract.py").exists(), "Open this notebook from the repo (notebooks/ or the repo root)."
sys.path.insert(0, str(PROJECT))

from src.pipeline.config import Config
from src.pipeline.runner import Pipeline, load_results, STAGES
from src.pipeline.triage import ontology_gap_terms, ParagraphSearch
from src.pipeline.evaluate import evaluate_paper, draft_gold_from_extraction
from src import extract as X

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 200)

cfg = Config(
    project_root=PROJECT,
    model="claude-sonnet-5-5",        # claude-haiku-5-5 = cheaper bulk runs; claude-opus-5-5 = hardest papers / evaluation
    max_output_tokens=16000,
    allow_live_calls=True,             # False = replay cached responses only (no API charges)
    min_confidence_for_auto_accept=0.85,
    parser="docling",                  # "docling" (layout model) or "pdfplumber" (old heuristics)
    docling={"do_ocr": False},         # see DOCLING_DEFAULTS in src/pipeline/docling_ingest.py
    table_vision="fallback",           # Claude re-reads table crops Docling couldn't structure ("always" / "never")
    figure_vision=False,               # True: Claude describes each figure (axes, samples, conditions; no values)
)
pl = Pipeline(cfg)
for n in cfg.notes:
    print("note:", n)
print("Ontology file:", cfg.ontology_file.name)
print("Claude:", pl.llm.status)
papers = pl.papers()
print(f"{len(papers)} PDFs found in {cfg.papers_dir} (including subfolders)")
if not papers:
    print("-> Put PDFs in data/publications/ (subfolders are fine), then re-run this cell.")

## 1. The ontology the extractor will use

The slice manifest (`ontology/slice_manifest.json`) turns subtrees of your ontology into the **only** class labels Claude may return (enums in the
structured-output schema). Warnings list missing roots and relations that have no object property in the ontology
yet. Both are to-dos for Protégé.

In [ ]:
summary = pl.profile.summary()
display(pd.DataFrame([{"group": g, **v} for g, v in summary["groups"].items()]))
display(pd.DataFrame([{"relation": k, "maps_to": v} for k, v in summary["relations"].items()]))
for w in summary["warnings"]:
    print("WARNING:", w)

In [ ]:
# The class tree as the pipeline sees it (like Protégé's hierarchy view)
print(pl.ontology.tree_text()[:4000])

## 2. Ingest and structure all papers (local, no API cost)

Docling converts each PDF once (the result is saved as `data/extractions/<paper>/docling.json`, with figure and table
crops in `crops/`); re-running reuses it. There is no single/two-column label any more: Docling's layout model detects
regions on every page and orders them, so mixed pages (full-width figure above two columns, etc.) are handled.

Checks before spending anything on the API:
* **confidence**: Docling's own grade per paper (`poor` / `fair` / `good` / `excellent`). Look at `poor`/`fair` papers first.
* **tables / figures**: counts per paper; `tables_need_vision` are tables Docling couldn't structure (Claude re-reads those).
  `figures_no_caption` are often logos or caption-less insets.
* **roles found**: `methods` and `results` should be present. If headings are missed, look at `sections`.
* **debug pages** (section 2.1): coloured boxes for every detected region. The fastest way to spot problems.

In [ ]:
run = pl.run_all(until="structure")
rows = []
for pid, r in run["results"].items():
    ing, st = r["ingest"], r["structure"]
    rows.append({
        "paper_id": pid, "pages": ing["source"]["page_count"],
        "status": ing.get("status"), "confidence": (ing.get("confidence") or {}).get("mean_grade"),
        "sections": len(st["sections"]), "paragraphs": len(st["paragraphs"]),
        "tables": len(st.get("tables", [])), "tables_need_vision": sum(t["needs_vision"] for t in st.get("tables", [])),
        "figures": len(st.get("figures", [])), "figures_no_caption": sum(not f["has_caption"] for f in st.get("figures", [])),
        "superscript_fixes": st.get("stats", {}).get("text_from_pdf_chars", 0),
        "roles": ", ".join(st["roles_found"]),
        "warnings": "; ".join(ing["warnings"] + st["warnings"]),
    })
structure_overview = pd.DataFrame(rows)
display(structure_overview)
print("Errors:", run["errors"] or "none")

In [ ]:
# Inspect one paper's sections and paragraphs
assert len(structure_overview), "No papers were processed: check the PDF count in the setup cell and the errors above."
PAPER = structure_overview.paper_id.iloc[0]   # or pick another paper_id from the table above
st = run["results"][PAPER]["structure"]
print("TITLE:", st["title"])
display(pd.DataFrame(st["sections"]))
display(pd.DataFrame(st["paragraphs"])[["id", "role", "kind", "page", "text"]].head(40))
display(pd.DataFrame(st.get("tables", [])))
display(pd.DataFrame(st.get("figures", [])))

### 2.1 See what was detected (debug pages, table and figure crops)

Each page is drawn with coloured boxes: blue text, magenta headings/title, green tables, red figures, orange captions,
purple formulas, grey references/footnotes. Labels give the paragraph ID used later as evidence.

In [ ]:
from IPython.display import Image, Markdown
from src.pipeline.docling_ingest import load_docling_document, render_debug_pages
from src.pipeline.io_utils import paper_id_from_filename

if cfg.parser == "docling":
    paper_dir = cfg.outputs_dir / PAPER
    pdf_path = next(p for p in papers if paper_id_from_filename(p) == PAPER)
    pngs = render_debug_pages(pdf_path, st, load_docling_document(paper_dir / "docling.json"), paper_dir / "debug")
    print(f"{len(pngs)} debug pages in {paper_dir / 'debug'}")
    for png in pngs[:4]:              # show the first pages; open the folder for the rest
        display(Image(filename=str(png), width=650))

In [ ]:
# Tables as Docling structured them (with the crop for comparison) and figure crops with their classification
if cfg.parser == "docling":
    paras = {p["id"]: p for p in st["paragraphs"]}
    for t in st.get("tables", [])[:5]:
        display(Markdown(f"**{t['label']}** (page {t['page']}, {t['rows']}x{t['cols']}, needs vision: {t['needs_vision']})"))
        if t.get("crop"):
            display(Image(filename=str(paper_dir / t["crop"]), width=500))
        display(Markdown(paras[t["paragraph_id"]]["text"]))
    for f in st.get("figures", [])[:6]:
        display(Markdown(f"**{f['label']}** (page {f['page']}) {f.get('classification')}: {f['caption'][:200]}"))
        if f.get("crop"):
            display(Image(filename=str(paper_dir / f["crop"]), width=350))

### 2.2 Ontology development aids (Murari-style, local)

Frequent technical phrases from the methods and results sections that match **no** class label or synonym in your
ontology. Use the list to decide which classes or synonyms to add in Protégé. It's a candidate list, not a verdict.

In [ ]:
structures = {pid: r["structure"] for pid, r in run["results"].items()}
display(pd.DataFrame(ontology_gap_terms(structures, pl.ontology, top=40)))

In [ ]:
# BM25 search across all papers' paragraphs (useful to check coverage before/after extraction)
search = ParagraphSearch(structures)
display(pd.DataFrame(search.search("annealing temperature carrier lifetime passivation", k=8)))

## 3. Cost check, then a pilot on 1–2 papers

The paper text is sent once per pass (three passes), but it is marked for **prompt caching**, so passes 2 and 3
read it from the cache at a fraction of the input price. Table pages are sent as images. The estimate below is rough
(about 4 characters per token). Vision calls: one per table Docling couldn't structure (or every table with
`table_vision="always"`), plus one per figure if `figure_vision=True`. Check current prices at docs.claude.com before large runs.

In [ ]:
est = []
for pid, r in run["results"].items():
    docs = X.build_documents(r["structure"], r["structure"]["title"], cfg.skip_sections, cfg.max_chars_per_call)
    chars = sum(d["chars"] for d in docs)
    est.append({"paper_id": pid, "chunks": len(docs), "doc_chars": chars, "approx_doc_tokens": chars // 4,
                "vision_calls": (sum(t["needs_vision"] for t in r["structure"].get("tables", [])) if cfg.table_vision == "fallback"
                                 else len(r["structure"].get("tables", [])) if cfg.table_vision == "always" else 0)
                                + (len(r["structure"].get("figures", [])) if cfg.figure_vision else 0)})
est = pd.DataFrame(est)
display(est)
print("System prompt (rules + ontology glossary): ~", len(pl.system) // 4, "tokens")
print("Total document tokens (one pass):", int(est.approx_doc_tokens.sum()))

In [ ]:
PILOT = papers[:1]          # start with one paper; read the results carefully before running all 50
pilot = pl.run_all(papers=PILOT, until="resolve")
print("Errors:", pilot["errors"] or "none")
display(pl.llm.usage_table())

### 3.1 Read the pilot output against the PDF

Look for: samples that should be split (different process histories), steps in the wrong order, values attached to
the wrong sample, wrong classes, and **things that are missing**. Fixes usually go into
`data/fewshot_examples.json` (add a positive and a *wrong* example), the profile, or the ontology. Re-running
then re-extracts only what changed.

In [ ]:
r = next(iter(pilot["results"].values()))   # the first pilot paper
p1, p2, p3 = r["pass1"], r["pass2"], r["pass3"]
print("Samples"); display(pd.DataFrame(p1["samples"])[["id", "name_in_paper", "ontology_class", "description", "material_ids", "confidence"]])
steps = pd.DataFrame([{**{k: s[k] for k in ("id", "ontology_class", "name_in_text", "step_order", "input_ids", "output_sample_ids")},
                       "parameters": "; ".join(f"{p['attribute_class']}={p['value_text']}" for p in s["parameters"])}
                      for s in p1["process_steps"]])
print("Process steps"); display(steps)
print("Unresolved notes:", p1["unresolved_notes"])

In [ ]:
cols = ["id", "technique_class", "property_class", "sample_ids", "value_text", "unit", "conditions", "data_origin", "polarity", "source_location", "confidence"]
print("Measurements"); display(pd.DataFrame(p2["measurements"])[cols])
print("Relations"); display(pd.DataFrame(p3["relations"])[["subject_id", "predicate", "object_id", "trend", "scope", "polarity", "support_type", "confidence"]] if p3["relations"] else "none")
print("Rejected candidates (kept on purpose)"); display(pd.DataFrame(p3["rejected_candidates"]))

## 4. Validation: what the code could and couldn't confirm

* `rejected`: no quote could be found in the paper, or an ID doesn't exist. Not exported.
* `needs_review`: anchored, but something deserves a human look (OTHER class, cited/computed value, hypothetical,
  number not found in the quoted paragraph, ontology gap, low confidence).
* `auto_accepted`: passed every check **and** confidence ≥ threshold. Still `verified_semantic_truth = False`.

In [ ]:
val = r["validate"]
display(pd.DataFrame(val["summary"]).pivot(index="kind", columns="status", values="count").fillna(0).astype(int))
problems = [{"kind": it["kind"], "id": it["id"], "status": it["status"], "issues": "; ".join(it["issues"] + it["notes"]),
             "quote": (it["evidence_anchors"][0]["quote"] if it["evidence_anchors"] else "")}
            for it in val["items"] if it["status"] != "auto_accepted"]
display(pd.DataFrame(problems))

## 5. Run all papers

Failures are collected per paper and don't stop the run. Re-running this cell resumes where it stopped: finished
stages are skipped and Claude calls replay from the cache.

In [ ]:
full = pl.run_all(until="resolve")
print("Papers done:", len(full["results"]), "| errors:", len(full["errors"]))
for k, v in full["errors"].items():
    print(" ", k, "->", v)
usage = pl.llm.usage_table()
if len(usage):
    print(usage.drop(columns=["label"]).groupby("cached").sum(numeric_only=True))

## 6. Export the knowledge graph

Writes to `data/graph/`:
* `nodes.jsonl`, `relationships.jsonl`: property graph in the same APOC JSON-lines format as `feder-graph/`. Every node and edge carries `paper_id`, `status`, `confidence`, `evidence`.
* `graph.ttl`: RDF typed with your ontology's class IRIs (can also be imported with Neo4j's neosemantics plugin).
* `review_queue.csv`: everything not auto-accepted, with the quote and page, plus empty `reviewer_decision` / `reviewer_comment` columns.
* `ontology_gaps.csv`: OTHER classes proposed by Claude, aggregated across papers. This is your to-do list for Protégé.
* `run_manifest.json`: config, software versions, ontology summary, and PDF hashes, for reproducibility.

Materials are merged across papers by (ontology class, normalized name). Samples, process steps and measurements stay
paper-specific.

In [ ]:
results = load_results(cfg.outputs_dir)   # everything on disk, including earlier runs
info = pl.export(results)
print(info)
display(pd.read_csv(cfg.graph_dir / "ontology_gaps.csv", encoding="utf-8-sig").head(30))
display(pd.read_csv(cfg.graph_dir / "review_queue.csv", encoding="utf-8-sig").head(20))

### 6.1 Load into Neo4j (optional)

Start Neo4j (`docker compose up -d`), check `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` in `.env`, then set
`LOAD_NEO4J = True`. This does the same as `python -m src.ontology_loader` (constraints + ontology class hierarchy)
followed by `python -m src.load_graph --replace`. All nodes get the label `KGNode` (unique `kg_key`) plus a type label
(`Sample`, `ProcessStep`, `Measurement`, …).

Example queries once loaded:

```cypher
// Process conditions behind lifetimes above 1 ms (experimental values only)
MATCH (m:Measurement)-[:MEASURED_ON]->(s:Sample)<-[:HAS_OUTPUT]-(p:ProcessStep)-[:HAS_PARAMETER]->(x:Parameter)
WHERE m.property = 'carrier lifetime' AND m.value_si > 1e-3 AND m.data_origin = 'experimental'
RETURN m.paper_id, s.name, p.name, x.attribute, x.value_text, m.value_text ORDER BY m.value_si DESC;

// Everything typed under a class, using the ontology hierarchy
MATCH (n)-[:INSTANCE_OF]->(:OntologyClass)-[:SUBCLASS_OF*0..]->(c:OntologyClass {name: 'deposition process'})
RETURN n.paper_id, n.name, n.ontology_class;

// Ontology gaps: instances Claude could not type
MATCH (n:KGNode) WHERE n.proposed_class IS NOT NULL RETURN n.proposed_class, count(*) AS n ORDER BY n DESC;
```

In [ ]:
LOAD_NEO4J = False
if LOAD_NEO4J:
    from src.ontology_loader import create_neo4j_schema
    from src.load_graph import driver_from_env, load_jsonl
    with driver_from_env() as driver:
        print(create_neo4j_schema(pl.ontology, driver))
        print(load_jsonl(driver, cfg.graph_dir / "nodes.jsonl", cfg.graph_dir / "relationships.jsonl", replace=True))

## 7. Evaluate against your hand-annotated papers

Without this step you can't tell whether a prompt or ontology change helped. Annotate 8–10 papers (see
`data/gold/README.md`), keep them out of the few-shot examples, and re-run this section after every change. Report
regressions as well as gains.

In [ ]:
# Write a draft gold file for a paper you are about to annotate (then correct it by hand and save as data/gold/<paper_id>.json)
DRAFT_FOR = None   # e.g. "Smith_2021_PECVD_SiNx"
if DRAFT_FOR:
    print(draft_gold_from_extraction(results[DRAFT_FOR]["resolve"], cfg.gold_dir / f"_draft_{DRAFT_FOR}.json"))

In [ ]:
scores = []
for gold_file in sorted(cfg.gold_dir.glob("*.json")):
    if gold_file.name.startswith("_"):
        continue
    pid = gold_file.stem
    if pid not in results or "resolve" not in results[pid]:
        print("No extraction for", pid); continue
    s = evaluate_paper(results[pid]["resolve"], json.loads(gold_file.read_text(encoding="utf-8")))["scores"]
    for k, v in s.items():
        if isinstance(v, dict):
            scores.append({"paper_id": pid, "item": k, **{m: v[m] for m in ("precision", "recall", "f1", "tp", "predicted", "gold")}})
        else:
            scores.append({"paper_id": pid, "item": k, "precision": v})
scores = pd.DataFrame(scores)
if len(scores):
    display(scores)
    display(scores.groupby("item")[["precision", "recall", "f1"]].mean())
else:
    print("No gold files yet. See data/gold/README.md.")

## 8. The iteration loop

1. **Read** the pilot output and the review queue against the PDFs.
2. **Fix the cause:**
   * wrong or missing classes → the ontology in Protégé, or `ontology/slice_manifest.json`
   * recurring judgment mistakes → a positive *and* a wrong example in `data/fewshot_examples.json`
   * layout/heading problems → check the debug pages (section 2.1); Docling options in `DOCLING_DEFAULTS`
     (`src/pipeline/docling_ingest.py`), e.g. `do_ocr` for scanned PDFs or `table_mode`
3. **Re-run.** Only affected stages re-run; unchanged Claude calls replay from the cache.
4. **Evaluate** on the gold papers and keep the change only if it helps.

Known limits: numbers shown only in figures are not extracted (by design). Scanned PDFs need OCR. Model
confidence is not calibrated. Cross-paper merging covers materials only; sample/process equivalence across papers is
a research question for later (e.g. similar recipes).